## 对比学习增强的 Transformer 参数消融实验

本 notebook 仅用于做参数实验：
- `lambda_cl` 扫描
- 增强强度对照（减半）
- 异常分数聚合方式对照（feature-mean vs feature-max）

注意：已删除/清空“单次完整训练+评估”的冗余单元格，避免你每次都先跑一遍慢训练。直接运行到函数定义完成后，运行最后的实验循环即可。


In [ ]:
from pathlib import Path
import os
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler

train_path = str(Path(os.environ.get("WADI_DATA_ROOT", "../data/WADI")) / "WADI_14days_new.csv")
test_path  = str(Path(os.environ.get("WADI_DATA_ROOT", "../data/WADI")) / "WADI_attackdataLABLE.csv")

assert os.path.exists(train_path), f"not found: {train_path}"
assert os.path.exists(test_path), f"not found: {test_path}"

train_df = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)

print("train_df:", train_df.shape)
print("test_raw:", test_raw.shape)

# 处理测试集表头与标签
test_raw.columns = test_raw.iloc[0]
test_raw = test_raw.iloc[1:].reset_index(drop=True)
if len(test_raw) >= 2:
    test_raw = test_raw.iloc[:-2]

label_col = "Attack LABLE (1:No Attack, -1:Attack)"
assert label_col in test_raw.columns, f"label column not found: {label_col}"

test_labels = test_raw[label_col].replace({1: 0, -1: 1}).astype(int).to_numpy()
test_df = test_raw.drop(columns=[label_col])

# drop meta
for c in ["Row ", "Date ", "Time"]:
    if c in test_df.columns:
        test_df = test_df.drop(columns=[c])
for c in ["Row", "Date", "Time"]:
    if c in train_df.columns:
        train_df = train_df.drop(columns=[c])

# 数值化
train_df = train_df.dropna(how="all").astype(float)
test_df = test_df.astype(float)

# NaN 清洗：删除全为 NaN 的列，其余用训练集中位数填充
all_nan_cols_train = train_df.columns[train_df.isna().all(axis=0)]
all_nan_cols_test  = test_df.columns[test_df.isna().all(axis=0)]
all_nan_cols = sorted(list(set(all_nan_cols_train).union(set(all_nan_cols_test))))

if len(all_nan_cols) > 0:
    print("Drop all-NaN columns:", len(all_nan_cols))
    train_df = train_df.drop(columns=all_nan_cols)
    test_df  = test_df.drop(columns=all_nan_cols)

train_median = train_df.median()
train_df = train_df.fillna(train_median)
test_df  = test_df.fillna(train_median)

print("train_df clean:", train_df.shape)
print("test_df  clean:", test_df.shape)
print("test_labels:", test_labels.shape, "anomaly count:", int(test_labels.sum()))

# 标准化（只用训练集）
scaler = StandardScaler()
train_data = scaler.fit_transform(train_df.values)
test_data  = scaler.transform(test_df.values)

train_data = np.nan_to_num(train_data, nan=0.0, posinf=0.0, neginf=0.0)
test_data  = np.nan_to_num(test_data,  nan=0.0, posinf=0.0, neginf=0.0)

D = train_data.shape[1]
print("feature dim D =", D)

# 构建 Dataset/DataLoader（与基线一致）
from torch.utils.data import Dataset, DataLoader

class SlidingWindowWithIndexDataset(Dataset):
    def __init__(self, data: np.ndarray, window_size: int, stride: int = 1):
        assert stride >= 1
        self.data = torch.tensor(data, dtype=torch.float32)
        self.window_size = window_size
        self.stride = stride

    def __len__(self):
        max_start = len(self.data) - self.window_size
        if max_start < 0:
            return 0
        return max_start // self.stride + 1

    def __getitem__(self, i: int):
        start = i * self.stride
        x = self.data[start : start + self.window_size]
        return x, start

window_size = 100
batch_size = 64
train_stride = 5

train_ds_train = SlidingWindowWithIndexDataset(train_data, window_size, stride=train_stride)
train_ds_eval  = SlidingWindowWithIndexDataset(train_data, window_size, stride=1)
test_ds        = SlidingWindowWithIndexDataset(test_data,  window_size, stride=1)

# Windows + Jupyter 环境下，num_workers>0 容易出现 worker 退出（你现在这个报错就是）。
# 这里默认用最稳的 0；如果你后续想提速，再手动改成 2 试试。
_num_workers = 0
_pin = bool(torch.cuda.is_available())

train_loader = DataLoader(
    train_ds_train,
    batch_size=batch_size,
    shuffle=True,
    drop_last=False,
    num_workers=_num_workers,
    pin_memory=_pin,
)
train_eval_loader = DataLoader(
    train_ds_eval,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=_num_workers,
    pin_memory=_pin,
)
test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=_num_workers,
    pin_memory=_pin,
)

xb, ib = next(iter(train_loader))
print("batch x:", xb.shape, "batch idx:", ib.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# ============ 时间序列增强（对比学习用） ============

def jitter(x: torch.Tensor, sigma: float = 0.01) -> torch.Tensor:
    if sigma <= 0:
        return x
    return x + torch.randn_like(x) * sigma


def scaling(x: torch.Tensor, sigma: float = 0.1) -> torch.Tensor:
    if sigma <= 0:
        return x
    B, L, D = x.shape
    scale = torch.randn(B, 1, D, device=x.device) * sigma + 1.0
    return x * scale


def time_mask(x: torch.Tensor, mask_ratio: float = 0.05) -> torch.Tensor:
    if mask_ratio <= 0:
        return x
    B, L, D = x.shape
    num_mask = int(L * mask_ratio)
    if num_mask <= 0:
        return x
    x_masked = x.clone()
    for b in range(B):
        idx = torch.randperm(L, device=x.device)[:num_mask]
        x_masked[b, idx, :] = 0.0
    return x_masked


def feature_mask(x: torch.Tensor, mask_ratio: float = 0.05) -> torch.Tensor:
    if mask_ratio <= 0:
        return x
    B, L, D = x.shape
    num_mask = int(D * mask_ratio)
    if num_mask <= 0:
        return x
    x_masked = x.clone()
    for b in range(B):
        idx = torch.randperm(D, device=x.device)[:num_mask]
        x_masked[b, :, idx] = 0.0
    return x_masked


def augment_ts(x: torch.Tensor) -> torch.Tensor:
    x1 = jitter(x, sigma=0.01)
    x1 = scaling(x1, sigma=0.1)
    x1 = time_mask(x1, mask_ratio=0.05)
    x1 = feature_mask(x1, mask_ratio=0.05)
    return x1

In [ ]:
# ============ 带对比学习模块的 Transformer（仅定义，不在这里实例化/训练） ============

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor):
        return self.dropout(x + self.pe[:, : x.size(1)])


class ContrastiveTransformer(nn.Module):
    def __init__(self, input_dim: int, d_model: int = 256, nhead: int = 8,
                 num_encoder_layers: int = 3, num_decoder_layers: int = 3,
                 dim_feedforward: int = 512, dropout: float = 0.1,
                 proj_dim: int = 128):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model=d_model, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_decoder_layers)
        self.output_proj = nn.Linear(d_model, input_dim)

        self.proj_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(inplace=True),
            nn.Linear(d_model, proj_dim),
        )

    def encode(self, x: torch.Tensor):
        x_emb = self.pos_enc(self.input_proj(x))
        h = self.encoder(x_emb)          # [B,L,d_model]
        h_pool = h.mean(dim=1)           # [B,d_model]
        z = self.proj_head(h_pool)       # [B,proj_dim]
        z = F.normalize(z, dim=1)
        return h, z

    def forward(self, x: torch.Tensor):
        h, _ = self.encode(x)
        tgt_emb = self.pos_enc(self.input_proj(x))
        dec_out = self.decoder(tgt_emb, h)
        return self.output_proj(dec_out)


In [ ]:
# ============ InfoNCE 对比损失 ============

def info_nce_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.1) -> torch.Tensor:
    """z1, z2: [B, d]，同一行是正对，batch 内其余为负样本"""
    B, _ = z1.shape
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    logits = (z1 @ z2.t()) / temperature  # [B,B]
    labels = torch.arange(B, device=z1.device, dtype=torch.long)
    loss1 = F.cross_entropy(logits, labels)
    loss2 = F.cross_entropy(logits.t(), labels)
    return 0.5 * (loss1 + loss2)

In [ ]:
# （已清空）原先的“单次联合训练”单元格已移除。
# 现在只在最后的实验循环中按不同参数重新初始化并训练模型。


In [ ]:
# （已清空）原先的“基于单次训练好的模型计算分数”单元格已移除。
# 实验循环里会按不同 score 模式（mean/max）自行计算。


In [ ]:
# （已清空）原先的“单次阈值 + 点级指标”单元格已移除。
# 实验循环会输出每组参数的阈值与 point-wise / point-adjust 指标。


In [ ]:
# ============ point-adjust（事件级）工具函数（只定义，不在此处计算） ============

def prf1(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return float(p), float(r), float(f1)


def point_adjust(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """对每个真实异常连续段，只要命中任意一点，就把整段预测置为 1。"""
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int).copy()
    n = len(y_true)
    i = 0
    while i < n:
        if y_true[i] == 1:
            j = i
            while j < n and y_true[j] == 1:
                j += 1
            if y_pred[i:j].max() == 1:
                y_pred[i:j] = 1
            i = j
        else:
            i += 1
    return y_pred


In [ ]:
# ============ 小规模消融实验：lambda、增强强度、score 方式（mean vs max） ============
# 说明：这段会“重新初始化模型并训练”，不会复用上面已经训练好的 model。

import time
from typing import Optional

def compute_point_scores_mode(
    model,
    loader,
    seq_len: int,
    feature_reduce: str = "mean",
    max_batches: Optional[int] = None,
    log_every: int = 100,
):
    """feature_reduce: 'mean' | 'max'；max_batches 用于快速近似评估

    返回：
    - scores: [seq_len] 每个时间点的重构误差聚合分数
    - count : [seq_len] 每个时间点被多少个窗口覆盖（用于修复近似评估时 th 被 0 拉低的问题）
    """
    model.eval()
    score_sum = np.zeros(seq_len, dtype=np.float64)
    count = np.zeros(seq_len, dtype=np.int64)

    use_amp = (device.type == "cuda")

    with torch.no_grad():
        for bi, (x, idx) in enumerate(loader, start=1):
            if max_batches is not None and bi > max_batches:
                break

            x = x.to(device, non_blocking=True)
            idx = idx.numpy()

            with torch.cuda.amp.autocast(enabled=use_amp):
                x_hat = model(x)
                se = (x_hat - x).pow(2)  # [B,L,D]
                if feature_reduce == "mean":
                    err = se.mean(dim=2)
                elif feature_reduce == "max":
                    err = se.max(dim=2).values
                else:
                    raise ValueError("feature_reduce must be 'mean' or 'max'")

            err = err.detach().cpu().numpy()  # [B,L]
            B, L = err.shape
            for b in range(B):
                s = int(idx[b])
                e = s + L
                score_sum[s:e] += err[b]
                count[s:e] += 1

            if log_every and (bi % log_every == 0):
                if max_batches is None:
                    print(f"  [scoring] batch {bi}...", flush=True)
                else:
                    print(f"  [scoring] batch {bi}/{max_batches}...", flush=True)

    scores = score_sum / np.maximum(count, 1)
    return scores, count


from typing import Optional

def train_contrastive(
    *,
    lambda_cl: float,
    temperature: float,
    epochs: int,
    aug_strength: float,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
    max_batches_per_epoch: Optional[int] = None,
    use_amp: bool = True,
    log_every: int = 0,
):
    """训练一个新的对比学习模型并返回模型。

    加速点：
    - AMP 混合精度（GPU 上通常明显提速）
    - 可选截断每个 epoch 的 batch 数（max_batches_per_epoch）

    日志：默认只在 epoch 结束打印一次；如需按 batch 打印可设置 log_every>0。
    """

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass

    # 局部增强：强度缩放（aug_strength=0.5 表示减半）
    def augment_ts_scaled(x: torch.Tensor) -> torch.Tensor:
        x1 = jitter(x, sigma=0.01 * aug_strength)
        x1 = scaling(x1, sigma=0.1 * aug_strength)
        x1 = time_mask(x1, mask_ratio=0.05 * aug_strength)
        x1 = feature_mask(x1, mask_ratio=0.05 * aug_strength)
        return x1

    m = ContrastiveTransformer(input_dim=D).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=weight_decay)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    recon_criterion = nn.MSELoss(reduction="mean")

    scaler = torch.cuda.amp.GradScaler(enabled=bool(use_amp and device.type == "cuda"))

    print(
        f"[train start] lambda={lambda_cl} aug={aug_strength} epochs={epochs} "
        f"max_batches/ep={max_batches_per_epoch} temp={temperature} lr={lr} amp={scaler.is_enabled()}",
        flush=True,
    )

    for ep in range(1, epochs + 1):
        m.train()
        ep_t0 = time.time()

        total_recon = 0.0
        total_cl = 0.0
        n = 0

        for bi, (x, _idx) in enumerate(train_loader, start=1):
            if max_batches_per_epoch is not None and bi > max_batches_per_epoch:
                break

            x = x.to(device, non_blocking=True)
            x1 = augment_ts_scaled(x)
            x2 = augment_ts_scaled(x)

            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
                x_hat = m(x)
                loss_recon = recon_criterion(x_hat, x)

                _, z1 = m.encode(x1)
                _, z2 = m.encode(x2)
                loss_cl = info_nce_loss(z1, z2, temperature=temperature)

                loss = loss_recon + lambda_cl * loss_cl

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

            bs = x.size(0)
            total_recon += float(loss_recon.item()) * bs
            total_cl += float(loss_cl.item()) * bs
            n += bs

            if log_every and (bi % log_every == 0):
                recon_avg = total_recon / max(n, 1)
                cl_avg = total_cl / max(n, 1)
                tot_avg = recon_avg + lambda_cl * cl_avg
                print(
                    f"  [batch] ep {ep:03d}/{epochs} bi {bi:04d} | recon={recon_avg:.6f} cl={cl_avg:.6f} total={tot_avg:.6f}",
                    flush=True,
                )

        sch.step()

        recon_avg = total_recon / max(n, 1)
        cl_avg = total_cl / max(n, 1)
        tot_avg = recon_avg + lambda_cl * cl_avg
        print(
            f"[epoch end] ep {ep:03d}/{epochs} | batches={bi} | recon={recon_avg:.6f} cl={cl_avg:.6f} total={tot_avg:.6f} "
            f"| lr={sch.get_last_lr()[0]:.2e} | {time.time()-ep_t0:.1f}s",
            flush=True,
        )

    return m


def eval_scores(
    *,
    model,
    feature_reduce: str,
    quantile: float = 0.95,
    max_eval_batches: Optional[int] = None,
    fast_eval_len: int = 50_000,
):
    """评估并返回阈值与指标。

    重要：如果你用 max_eval_batches 做“快速评估”，顺序 DataLoader 只能覆盖序列开头很小一段，
    会导致 train_cov/test_cov 很低，进而不同超参看起来几乎一样。

    解决：当 max_eval_batches 不为 None 时，把评估限制在一个连续子序列（默认前 fast_eval_len 点），
    让覆盖率接近 1，这样少量 batch 也能得到有区分度的消融结果。
    """

    def _make_loader_for_prefix(data_np: np.ndarray, *, length: int):
        length = int(min(length, len(data_np)))
        ds = SlidingWindowWithIndexDataset(data_np[:length], window_size, stride=1)
        return DataLoader(ds, batch_size=batch_size, shuffle=False, drop_last=False), length

    if max_eval_batches is None:
        # 全量评估：覆盖率自然接近 1
        train_loader_eval = train_eval_loader
        test_loader_eval = test_loader
        train_len = len(train_data)
        test_len = len(test_data)
        test_labels_eval = test_labels
    else:
        # 快速评估：只评估前 fast_eval_len 点，避免 coverage 过低
        train_loader_eval, train_len = _make_loader_for_prefix(train_data, length=fast_eval_len)
        test_loader_eval, test_len = _make_loader_for_prefix(test_data, length=fast_eval_len)
        test_labels_eval = test_labels[:test_len]

    train_scores, train_count = compute_point_scores_mode(
        model,
        train_loader_eval,
        seq_len=train_len,
        feature_reduce=feature_reduce,
        max_batches=max_eval_batches,
    )
    test_scores, test_count = compute_point_scores_mode(
        model,
        test_loader_eval,
        seq_len=test_len,
        feature_reduce=feature_reduce,
        max_batches=max_eval_batches,
    )

    valid_train = train_count > 0
    if valid_train.any():
        th = float(np.quantile(train_scores[valid_train], quantile))
    else:
        th = 0.0

    pred = np.zeros_like(test_scores, dtype=int)
    valid_test = test_count > 0
    pred[valid_test] = (test_scores[valid_test] > th).astype(int)

    p, r, f1, _ = precision_recall_fscore_support(test_labels_eval, pred, average="binary", zero_division=0)
    pa_pred = point_adjust(test_labels_eval, pred)
    pa_p, pa_r, pa_f1, _ = precision_recall_fscore_support(test_labels_eval, pa_pred, average="binary", zero_division=0)

    return {
        "th": th,
        "P": float(p),
        "R": float(r),
        "F1": float(f1),
        "PA_P": float(pa_p),
        "PA_R": float(pa_r),
        "PA_F1": float(pa_f1),
        "train_cov": float(valid_train.mean()),
        "test_cov": float(valid_test.mean()),
        "eval_len": int(test_len),
    }


# ============ 消融实验配置（改成“预算内”版本：4-5 小时可跑完） ============
# 关键思路：
# - 训练：每个 epoch 只跑固定数量 batch（近似训练），并启用 AMP
# - 评估：只在一个前缀子序列上做打分/阈值/指标，并限制评估 batch（快速近似）
# - 复用：同一模型的 score=mean/max 不再重复训练

EXP_EPOCHS = 8
TEMP = 0.2
Q = 0.95

# 训练预算：每个 epoch 只跑这么多 batch（越大越准，但越慢）
# 经验：在 WADI 上先用 300~600 做消融就很够用
MAX_BATCHES_PER_EPOCH = 450

# 评估预算：只跑这么多 batch（越大越准，但越慢）
MAX_EVAL_BATCHES = 900

# 快速评估时只看前缀长度（避免 coverage 太低导致阈值不稳定）
FAST_EVAL_LEN = 120_000

# lambda：从“6 个”减到“4 个”先粗扫；如有需要再围绕最优细扫
lambda_list = [0.0, 0.05, 0.1, 0.2]

runs = []
start_time = time.time()

# (实验1) 扫 lambda（增强强度=1.0），同一模型同时评估 mean/max
for lam in lambda_list:
    m = train_contrastive(
        lambda_cl=lam,
        temperature=TEMP,
        epochs=EXP_EPOCHS,
        aug_strength=1.0,
        max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
        use_amp=True,
    )
    for score_mode in ["mean", "max"]:
        met = eval_scores(
            model=m,
            feature_reduce=score_mode,
            quantile=Q,
            max_eval_batches=MAX_EVAL_BATCHES,
            fast_eval_len=FAST_EVAL_LEN,
        )
        runs.append({"exp": "lambda_sweep", "lambda": lam, "aug": 1.0, "score": score_mode, **met})
        print(f"lambda={lam:<4} score={score_mode:<4} | F1={met['F1']:.4f} | PA_F1={met['PA_F1']:.4f} | cov={met['test_cov']:.3f}")

# (实验2) 增强强度对照（固定 lambda=0.1），同一模型同时评估 mean/max
for aug in [1.0, 0.5]:
    m = train_contrastive(
        lambda_cl=0.1,
        temperature=TEMP,
        epochs=EXP_EPOCHS,
        aug_strength=aug,
        max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
        use_amp=True,
    )
    for score_mode in ["mean", "max"]:
        met = eval_scores(
            model=m,
            feature_reduce=score_mode,
            quantile=Q,
            max_eval_batches=MAX_EVAL_BATCHES,
            fast_eval_len=FAST_EVAL_LEN,
        )
        runs.append({"exp": "aug_strength", "lambda": 0.1, "aug": aug, "score": score_mode, **met})
        print(f"aug={aug:<3} score={score_mode:<4} | F1={met['F1']:.4f} | PA_F1={met['PA_F1']:.4f} | cov={met['test_cov']:.3f}")

elapsed = time.time() - start_time
print(f"\nAll runs finished. elapsed={elapsed/60:.1f} min, total records={len(runs)}")

try:
    import pandas as pd
    df = pd.DataFrame(runs)
    display(df.sort_values(["exp", "PA_F1"], ascending=[True, False]))
except Exception as e:
    print("Could not build DataFrame:", e)
    print(runs)
